In [28]:
# pip install duckdb
# poetry add duckdb
# uv add duckdb

In [29]:
import duckdb

## Set up a database and load the raw data into a table

We're going to create a DuckDB database to persist the data we have extracted from the CSV files and then take it through a few transformation steps to finally generate some useful analytics over the data.

In [30]:
from pathlib import Path
LAND_REGISTRY_DATABASE_PATH = Path("../databases/land_registry.db")

In [31]:
if LAND_REGISTRY_DATABASE_PATH.exists():
    LAND_REGISTRY_DATABASE_PATH.unlink()
else:
    LAND_REGISTRY_DATABASE_PATH.parent.mkdir(parents=True, exist_ok=True)

In [32]:
db = duckdb.connect(LAND_REGISTRY_DATABASE_PATH)
# db = duckdb.connect(":memory:")

In [33]:
db.sql("SELECT * from information_schema.tables")

┌───────────────┬──────────────┬────────────┬────────────┬──────────────────────────────┬──────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬────────────────────┬──────────┬───────────────┬───────────────┐
│ table_catalog │ table_schema │ table_name │ table_type │ self_referencing_column_name │ reference_generation │ user_defined_type_catalog │ user_defined_type_schema │ user_defined_type_name │ is_insertable_into │ is_typed │ commit_action │ TABLE_COMMENT │
│    varchar    │   varchar    │  varchar   │  varchar   │           varchar            │       varchar        │          varchar          │         varchar          │        varchar         │      varchar       │ varchar  │    varchar    │    varchar    │
└───────────────┴──────────────┴────────────┴────────────┴──────────────────────────────┴──────────────────────┴───────────────────────────┴──────────────────────────┴────────────────────────┴────────────────────┴──────────┴─────

## Load raw data into a table

In [34]:
db.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [35]:
DATA_PATH = Path("../data/land_registry/")

In [36]:
db.sql(
    f"""
    CREATE TABLE IF NOT EXISTS bronze.price_paid_raw AS
    SELECT
    column00 AS 'id',
    column01 AS 'price',
    column02 AS 'date',
    column03 AS 'postcode',
    column04 AS 'property_type',
    column05 AS 'old_new',
    column06 AS 'duration',
    column07 AS 'paon',
    column08 AS 'saon',
    column09 AS 'street',
    column10 AS 'locale',
    column11 AS 'town_city',
    column12 AS 'district',
    column13 AS 'county',
    column14 AS 'ppd_category',
    column15 AS 'record_type',
    FROM read_csv('{DATA_PATH / "pp-*.csv"}');
    """
)

In [37]:
db.sql("SELECT COUNT (*) FROM bronze.price_paid_raw")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     10002480 │
└──────────────┘

## Add features

In [38]:
db.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [39]:
db.sql(
    f"""
    CREATE OR REPLACE VIEW silver.price_paid_cleaned AS
    SELECT
    id,
    price,
    date,
    year(date) AS 'year_of_sale',
    date_trunc('month', date) AS 'month_of_sale',
    CASE property_type
          WHEN 'D' THEN 'Detached'
          WHEN 'S' THEN 'Semi-Detached'
          WHEN 'T' THEN 'Terraced'
          WHEN 'F' THEN 'Flat'
          WHEN 'O' THEN 'Other'
          ELSE property_type
    END AS 'property_type',
    postcode,
    paon,
    saon,
    street,
    locale,
    town_city,
    district,
    county
    FROM bronze.price_paid_raw;
    """
)

In [40]:
db.sql(
    """
    SELECT
    * 
    FROM silver.price_paid_cleaned LIMIT 10
    """
)

┌────────────────────────────────────────┬────────┬─────────────────────┬──────────────┬─────────────────────┬───────────────┬──────────┬────────────┬─────────┬─────────────────┬────────────────┬─────────────┬───────────────────────┬───────────────────────┐
│                   id                   │ price  │        date         │ year_of_sale │    month_of_sale    │ property_type │ postcode │    paon    │  saon   │     street      │     locale     │  town_city  │       district        │        county         │
│                varchar                 │ int64  │      timestamp      │    int64     │      timestamp      │    varchar    │ varchar  │  varchar   │ varchar │     varchar     │    varchar     │   varchar   │        varchar        │        varchar        │
├────────────────────────────────────────┼────────┼─────────────────────┼──────────────┼─────────────────────┼───────────────┼──────────┼────────────┼─────────┼─────────────────┼────────────────┼─────────────┼─────────────────

## Projecting insights

In [41]:
db.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [42]:
db.sql(
    f"""
    CREATE OR REPLACE VIEW gold.average_price_by_month_and_property_type AS
    SELECT
    property_type,
    month_of_sale,
    MEDIAN(price) AS median_price
    FROM silver.price_paid_cleaned
    WHERE property_type <> 'Other'
    GROUP BY ALL
    ORDER BY month_of_sale, property_type;
    """
)

In [43]:
db.sql("FROM gold.average_price_by_month_and_property_type")

┌───────────────┬─────────────────────┬──────────────┐
│ property_type │    month_of_sale    │ median_price │
│    varchar    │      timestamp      │    double    │
├───────────────┼─────────────────────┼──────────────┤
│ Detached      │ 2016-01-01 00:00:00 │     300000.0 │
│ Flat          │ 2016-01-01 00:00:00 │     198000.0 │
│ Semi-Detached │ 2016-01-01 00:00:00 │     185000.0 │
│ Terraced      │ 2016-01-01 00:00:00 │     166995.0 │
│ Detached      │ 2016-02-01 00:00:00 │     299950.0 │
│ Flat          │ 2016-02-01 00:00:00 │     194200.0 │
│ Semi-Detached │ 2016-02-01 00:00:00 │     184995.0 │
│ Terraced      │ 2016-02-01 00:00:00 │     165000.0 │
│ Detached      │ 2016-03-01 00:00:00 │     305000.0 │
│ Flat          │ 2016-03-01 00:00:00 │     192000.0 │
│  ·            │          ·          │         ·    │
│  ·            │          ·          │         ·    │
│  ·            │          ·          │         ·    │
│ Semi-Detached │ 2025-10-01 00:00:00 │     270000.0 │
│ Terraced

## Inspect contents of database

In [44]:
db.sql("SELECT * from information_schema.tables")

┌───────────────┬──────────────┬──────────────────────────────────────────┬────────────┬──────────────────────────────┬──────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬────────────────────┬──────────┬───────────────┬───────────────┐
│ table_catalog │ table_schema │                table_name                │ table_type │ self_referencing_column_name │ reference_generation │ user_defined_type_catalog │ user_defined_type_schema │ user_defined_type_name │ is_insertable_into │ is_typed │ commit_action │ TABLE_COMMENT │
│    varchar    │   varchar    │                 varchar                  │  varchar   │           varchar            │       varchar        │          varchar          │         varchar          │        varchar         │      varchar       │ varchar  │    varchar    │    varchar    │
├───────────────┼──────────────┼──────────────────────────────────────────┼────────────┼──────────────────────────────┼────────────────────

## Inspect the files

In [45]:
database_files = list(LAND_REGISTRY_DATABASE_PATH.parent.glob("*.db"))
csv_files = list(DATA_PATH.glob("*.csv"))

In [46]:
database_files

[PosixPath('../databases/land_registry.db')]

In [47]:
import polars as pl

files_to_analyse = pl.DataFrame(
    { 
        "file_name": [f.name for f in database_files + csv_files],
        "file_size": [f.stat().st_size for f in database_files + csv_files],
        "file_type": [f.suffix for f in database_files + csv_files]
    }
)

In [48]:
files_to_analyse

file_name,file_size,file_type
str,i64,str
"""land_registry.db""",439365632,""".db"""
"""pp-2021.csv""",223121026,""".csv"""
"""pp-2018.csv""",180162507,""".csv"""
"""pp-2017.csv""",185326793,""".csv"""
"""pp-2025.csv""",139672841,""".csv"""
…,…,…
"""pp-2019.csv""",176072601,""".csv"""
"""pp-2024.csv""",161058370,""".csv"""
"""pp-2016.csv""",181579035,""".csv"""


In [49]:
(
    files_to_analyse
    .group_by("file_type")
    .agg(pl.col("file_size").sum().alias("total_file_size"))
    .with_columns((pl.col("total_file_size") / (1024 * 1024 * 1024)).alias("total_file_size_gb"))
)

file_type,total_file_size,total_file_size_gb
str,i64,f64
""".db""",439365632,0.409191
""".csv""",1740737180,1.621188


In [50]:
# Calculate total sizes for .db and .csv files
db_size = files_to_analyse.filter(pl.col("file_type") == ".db")["file_size"].sum()
csv_size = files_to_analyse.filter(pl.col("file_type") == ".csv")["file_size"].sum()

# Calculate compression percentage
compression_percentage = (db_size / csv_size) * 100
print(f"DuckDB database is {compression_percentage:.1f}% of the original size of the CSV files.")

DuckDB database is 25.2% of the original size of the CSV files.


## Add visualisation over the analytics

Finally we convert the Gold `average_price_by_month_and_property_type` view to a Polars dataframe so we can visualise the data on a line chart using the Plotly Express charting package.

In [51]:
import plotly.express as px

fig = px.line(
    db.sql("FROM gold.average_price_by_month_and_property_type").pl(),
    x="month_of_sale",
    y="median_price",
    color="property_type",
    line_dash="property_type",
    color_discrete_sequence=["#8CC600", "black", "darkgrey", "#EE7423"],
    line_dash_sequence=["solid", "dash", "dot", "dashdot"],
    title="Median Price by Month and Property Type"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Median Price",
    legend_title="Property Type",
)

fig.show()

## Write to File

In [52]:
from pathlib import Path
Path("../data/price_paid_insights").mkdir(parents=True, exist_ok=True)

db.sql(
    """
    COPY gold.average_price_by_month_and_property_type
    TO '../data/price_paid_insights/annual_price_by_property_type.parquet'
    (FORMAT PARQUET)
    """
)

In [53]:
# Close DuckDB connection
db.close()